# LLM-Based Question Answering Chatbot

This is a cleaned version of the original notebook, organized around its document-question-answering workflow.

**Pipeline:** PDF text extraction → section-aware chunking → embedding generation → FAISS cosine-similarity index → hybrid retrieval → answer generation from retrieved context.

Exploratory tests, repeated experiments, and one-off debugging cells have been removed.

## 1. Configuration and imports

Confirm that the PDF exists at `PDF_PATH`, Ollama is running, and both configured models are available. The original notebook installed FAISS in the project's virtual environment; this cell adds that environment's packages to the notebook kernel.

In [1]:
from pathlib import Path
import sys
import re
import json
import time
import requests
import numpy as np

PROJECT_DIR = Path("/jupyter/sods.user28/llm_qa_chatbot_builder")
PDF_PATH = PROJECT_DIR / "test_document.pdf"
EXTRACTED_TEXT_PATH = PROJECT_DIR / "extracted_text.txt"
CHUNKS_PATH = PROJECT_DIR / "section_chunks.json"
EMBEDDINGS_PATH = PROJECT_DIR / "section_embeddings.npy"
INDEX_PATH = PROJECT_DIR / "faiss_section_cosine_index.bin"

OLLAMA_URL = "http://localhost:11434"
EMBEDDING_MODEL = "nomic-embed-text"
GENERATION_MODEL = "llama3.1:8b-instruct-fp16"
VENV_SITE_PACKAGES = Path("/jupyter/sods.user28/llm_qa_chatbot_builder/.venv/lib/python3.9/site-packages")

if VENV_SITE_PACKAGES.exists() and str(VENV_SITE_PACKAGES) not in sys.path:
    sys.path.insert(0, str(VENV_SITE_PACKAGES))

import faiss

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

print("Project directory:", PROJECT_DIR)
print("PDF:", PDF_PATH)
print("FAISS version:", faiss.__version__)

Project directory: /jupyter/sods.user28/llm_qa_chatbot_builder
PDF: /jupyter/sods.user28/llm_qa_chatbot_builder/test_document.pdf
FAISS version: 1.13.0


## 2. Extract text from the PDF

Text is extracted page by page with `pypdf` and saved as UTF-8 text.

In [2]:
from pypdf import PdfReader

reader = PdfReader(str(PDF_PATH))
document_text = "\n".join(page.extract_text() or "" for page in reader.pages)

if not document_text.strip():
    raise ValueError("No text was extracted from the PDF. Check whether it is scanned or text-based.")

EXTRACTED_TEXT_PATH.write_text(document_text, encoding="utf-8")

print("Pages:", len(reader.pages))
print("Characters extracted:", len(document_text))
print("Saved extracted text to:", EXTRACTED_TEXT_PATH)

Pages: 187
Characters extracted: 298115
Saved extracted text to: /jupyter/sods.user28/llm_qa_chatbot_builder/extracted_text.txt


## 3. Create section-aware chunks

The original workflow detects numbered section headings, removes heading-only sections, then splits each section into chunks of 200 words with 40 words of overlap. Each chunk keeps its section ID.

In [3]:
def split_into_sections(text):
    heading_pattern = r"(?m)^\d+(?:\.\d+)*\s+[A-Z][^\n]*$"
    matches = list(re.finditer(heading_pattern, text))
    sections = []

    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        section = text[start:end].strip()
        if section:
            sections.append(section)

    return [section for section in sections if len(section.splitlines()) > 1]


def chunk_sections(sections, chunk_size=200, overlap=40):
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Use chunk_size > overlap >= 0.")

    all_chunks = []
    step = chunk_size - overlap

    for section_id, section in enumerate(sections):
        words = section.split()
        start = 0

        while start < len(words):
            chunk_text = " ".join(words[start:start + chunk_size])
            if chunk_text:
                all_chunks.append({"section_id": section_id, "text": chunk_text})
            if start + chunk_size >= len(words):
                break
            start += step

    return all_chunks


sections = split_into_sections(document_text)
chunks = chunk_sections(sections, chunk_size=200, overlap=40)

if not chunks:
    raise ValueError("No section-aware chunks were created. Check the PDF's section-heading format.")

CHUNKS_PATH.write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8")

print("Sections detected:", len(sections))
print("Chunks created:", len(chunks))
print("Saved chunks to:", CHUNKS_PATH)
print("\nFirst chunk preview:\n")
print(chunks[0]["text"][:700])

Sections detected: 199
Chunks created: 394
Saved chunks to: /jupyter/sods.user28/llm_qa_chatbot_builder/section_chunks.json

First chunk preview:

1 Introduction Introduction to Artificial Intelligence By Nikhil Sharma, Josh Hug, Jacky Liang, and Henry Zhu. Additional Contributions by Regina Wang, Peyrin Kao, Matei Gardea, Danial Toktarbayev, Catherine Chu, Samantha Huang, Saathvik Selvan, Pranav Muralikrishnan, and Wesley Zheng. Website Established By Wesley Zheng. Special Thanks to Ashley Chiu. This is the textbook for CS 188: Introduction to Artificial Intelligence at UC Berkeley. It provides an introduction to the full range of topics studied in artificial intelligence, with emphasis on the “core competences” of intelligent systems - problem solving, reasoning, decision making, and learning - and on the logical and probabilistic fo


## 4. Generate embeddings

The `nomic-embed-text` model creates an embedding vector for each chunk. Requests are batched and the resulting vectors are saved as a NumPy array.

In [4]:
def ollama_embed(texts, batch_size=32):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        response = requests.post(
            f"{OLLAMA_URL}/api/embed",
            json={"model": EMBEDDING_MODEL, "input": batch},
            timeout=300
        )
        response.raise_for_status()
        batch_embeddings = response.json().get("embeddings", [])
        if len(batch_embeddings) != len(batch):
            raise RuntimeError(
                f"Expected {len(batch)} embeddings, received {len(batch_embeddings)}."
            )
        all_embeddings.extend(batch_embeddings)
        print(f"Embedded {min(start + len(batch), len(texts))}/{len(texts)} chunks")

    return np.asarray(all_embeddings, dtype="float32")


chunk_texts = [chunk["text"] for chunk in chunks]
start_time = time.time()
embeddings = ollama_embed(chunk_texts)
np.save(EMBEDDINGS_PATH, embeddings)

print("Embedding shape:", embeddings.shape)
print("Saved embeddings to:", EMBEDDINGS_PATH)
print(f"Elapsed time: {time.time() - start_time:.1f} seconds")

Embedded 32/394 chunks
Embedded 64/394 chunks
Embedded 96/394 chunks
Embedded 128/394 chunks
Embedded 160/394 chunks
Embedded 192/394 chunks
Embedded 224/394 chunks
Embedded 256/394 chunks
Embedded 288/394 chunks
Embedded 320/394 chunks
Embedded 352/394 chunks
Embedded 384/394 chunks
Embedded 394/394 chunks
Embedding shape: (394, 768)
Saved embeddings to: /jupyter/sods.user28/llm_qa_chatbot_builder/section_embeddings.npy
Elapsed time: 14.8 seconds


## 5. Build the FAISS cosine-similarity index

Chunk vectors are L2-normalized and indexed with `IndexFlatIP`. For normalized vectors, inner-product search corresponds to cosine similarity.

In [5]:
vectors = np.load(EMBEDDINGS_PATH).astype("float32")

if vectors.ndim != 2 or len(vectors) != len(chunks):
    raise ValueError("The embedding array and chunk list do not match.")

faiss.normalize_L2(vectors)
index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)
faiss.write_index(index, str(INDEX_PATH))

print("FAISS index saved to:", INDEX_PATH)
print("Indexed chunks:", index.ntotal)
print("Embedding dimensions:", index.d)

FAISS index saved to: /jupyter/sods.user28/llm_qa_chatbot_builder/faiss_section_cosine_index.bin
Indexed chunks: 394
Embedding dimensions: 768


## 6. Retrieve context and generate an answer

For each question, the notebook embeds the question, retrieves candidates from FAISS, reranks retrieved chunks using semantic similarity and query-term overlap, and asks the language model to answer only from the retrieved context.

The prompt discourages unsupported answers, but this cannot guarantee that a language model will never hallucinate. The retrieved context is printed so answers can be checked against the source.

In [6]:
def tokenize(text):
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    return re.findall(r"\b[a-z0-9]+\b", text.lower())


STOPWORDS = {
    "what", "does", "stand", "for", "is", "are", "the",
    "a", "an", "in", "of", "to", "and", "how", "why",
    "with", "on", "it", "this", "that"
}


def ask_question(question, top_k=5, candidate_k=20, show_context=True):
    if not question.strip():
        raise ValueError("Please enter a question.")
    if index.ntotal == 0:
        raise RuntimeError("The FAISS index is empty.")

    response = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": EMBEDDING_MODEL, "input": question},
        timeout=120
    )
    response.raise_for_status()

    query_vector = np.asarray(
        response.json()["embeddings"][0], dtype="float32"
    ).reshape(1, -1)
    faiss.normalize_L2(query_vector)

    k = min(candidate_k, index.ntotal)
    scores, indices = index.search(query_vector, k)
    semantic_scores = {
        int(idx): float(score)
        for idx, score in zip(indices[0], scores[0])
        if idx >= 0
    }

    query_terms = set(tokenize(question)) - STOPWORDS
    ranked = []

    for idx, semantic_score in semantic_scores.items():
        content = chunks[idx]["text"]
        content_terms = set(tokenize(content))
        overlap = len(query_terms & content_terms) / max(len(query_terms), 1)
        combined_score = 0.6 * semantic_score + 0.4 * overlap
        ranked.append((combined_score, idx, semantic_score, overlap))

    ranked.sort(key=lambda item: item[0], reverse=True)
    selected = ranked[:min(top_k, len(ranked))]

    if not selected:
        raise RuntimeError("No relevant chunks were retrieved.")

    context = "\n\n".join(chunks[idx]["text"] for _, idx, _, _ in selected)
    prompt = f""""Answer the question using only the context below.
If the context does not provide enough information, say so.
Give a concise answer.

Context:
{context}

Question: {question}
Answer:"""

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": GENERATION_MODEL,
            "prompt": prompt,
            "stream": False
        },
        timeout=300
    )
    response.raise_for_status()
    answer = response.json().get("response", "").strip()

    print(f"Question: {question}\n")
    print(f"Answer: {answer}")

    return None

## 7. Ask questions

Change the question text to query the PDF.

In [7]:
ask_question("What does PEAS stand for?")

Question: What does PEAS stand for?

Answer: PEAS stands for Performance Measure, Environment, Actuators, and Sensors.


In [29]:
ask_question("What is a reflex agent in artificial intelligence?")

Question: What is a reflex agent in artificial intelligence?

Answer: A reflex agent is a type of agent that doesn't think about the consequences of its actions but selects an action based solely on the current state of the world.


In [30]:
ask_question("What is a search problem?")

Question: What is a search problem?

Answer: A search problem consists of the following elements:

• A state space - The set of all possible states that are possible in your given world
• A set of actions available in each state
• A transition model - Outputs the next state when a specific action is taken at current state
• An action cost - Incurred when moving from one state to another after applying an action
• A start state - The state in which an agent exists initially
• A goal test - A function that takes a state as input, and determines whether it is a goal state

Fundamentally, a search problem is solved by first considering the start state, then exploring the state space using the action and transition and cost methods, iteratively computing children of various states until we arrive at a goal state.


In [31]:
ask_question("What is a rational agent in artificial intelligence?")

Question: What is a rational agent in artificial intelligence?

Answer: A rational agent in artificial intelligence is an entity that has goals or preferences and tries to perform a series of actions that yield the best/optimal expected outcome given these goals.


In [32]:
ask_question("What is the difference between Breadth-First Search (BFS) and Depth-First Search (DFS)?")

Question: What is the difference between Breadth-First Search (BFS) and Depth-First Search (DFS)?

Answer: Breadth-First Search (BFS) selects the shallowest frontier node for expansion, whereas Depth-First Search (DFS) selects the deepest frontier node for expansion.


In [33]:
ask_question("What are the components of a search problem in artificial intelligence?")


Question: What are the components of a search problem in artificial intelligence?

Answer: A search problem in artificial intelligence consists of the following elements:

1. A state space - The set of all possible states that are possible in the given world.
2. A set of actions available in each state.
3. A transition model - Outputs the next state when a specific action is taken at the current state.
4. An action cost - Incurred when moving from one state to another after applying an action.
5. A start state - The state in which an agent exists initially.
6. A goal test - A function that takes a state as input, and determines whether it is a goal state.


In [34]:
ask_question("What is a neural network?")

Question: What is a neural network?

Answer: A neural network is an artificial neural network, which is an extension of a multi-layer perceptron that uses added non-linearities to make the network more expressive.


In [35]:
ask_question("What is an artificial neural network, and how do non-linearities make it more expressive?")

Question: What is an artificial neural network, and how do non-linearities make it more expressive?

Answer: An artificial neural network is introduced in the context, where a multi-layer feedforward neural network is described. Non-linearities, specifically added to each perceptron node, make the network as a whole non-linear and more expressive. Without these non-linearities, the network would simply be a composition of linear functions and hence still linear.


In [36]:
ask_question("Who invented the telephone?")

Question: Who invented the telephone?

Answer: The context does not provide enough information to answer the question about who invented the telephone.


In [38]:
ask_question("What is the minimax algorithm?")

Question: What is the minimax algorithm?

Answer: The minimax algorithm is a zero-sum-game algorithm that assumes the opponent behaves optimally and will always perform the move that is worst for the agent. It maximizes the utility over the children of nodes controlled by the agent, while minimizing over the children of nodes controlled by the opponent.


In [39]:
ask_question("What is the difference between supervised learning and unsupervised learning?")

Question: What is the difference between supervised learning and unsupervised learning?

Answer: In supervised learning, the algorithm infers a relationship between input data and corresponding output data to predict outputs for new, unseen input data. In contrast, unsupervised learning deals with input data that doesn't have any corresponding output data and recognizes inherent structure between or within datapoints, grouping and/or processing them accordingly.


## Notes

- Run the notebook from top to bottom after confirming the PDF path and Ollama models.
- Section detection depends on numbered headings matching the regular expression in Section 3.
- The notebook does not guarantee hallucination-free answers; check generated answers against the retrieved context.